# Does Regularization Shape Prunability? — experiment runner
Settings → Accelerator: **GPU T4 x2**, Internet: **On**.
**Add Input → your Kaggle dataset containing `cifar-10-python.tar.gz`** (md5 c58f30108f718f92721af3b95e74349a) to skip the 35-minute download.
Each 100-epoch ResNet-18 run takes ~25 min on a T4; two runs execute in parallel, one per GPU.
Use **Save Version → Save & Run All (Commit)** so outputs in `/kaggle/working` are kept after the session ends.

In [ ]:
import os, subprocess, glob, shutil
REPO = 'https://github.com/WHITELOTUS0/dropout-prunability.git'
if not os.path.exists('/kaggle/working/code'):
    subprocess.run(['git', 'clone', REPO, '/kaggle/working/code'], check=True)
os.chdir('/kaggle/working/code')
!git pull -q; nvidia-smi --query-gpu=name,memory.total --format=csv; python -c "import torch;print(torch.__version__, torch.cuda.device_count())"
# Use an attached Kaggle dataset copy of CIFAR-10 if present (the Toronto server downloads at ~75 kB/s, 35 min)
os.makedirs('data', exist_ok=True)
tars = glob.glob('/kaggle/input/**/cifar-10-python.tar.gz', recursive=True)
if tars and not os.path.exists('data/cifar-10-python.tar.gz'):
    shutil.copy(tars[0], 'data/cifar-10-python.tar.gz'); print('using dataset copy:', tars[0])
else:
    print('no attached CIFAR-10 tarball found; torchvision will download it (slow)' if not tars else 'tarball already present')
# Resume: reuse trained models from a previous version of this notebook attached as Input (Add Input -> Your Work)
os.makedirs('runs', exist_ok=True); n = 0
for mp in glob.glob('/kaggle/input/**/runs/p*_s*/model.pt', recursive=True):
    d = os.path.basename(os.path.dirname(mp)); dst = os.path.join('runs', d)
    if os.path.exists(os.path.join(dst, 'model.pt')): continue
    os.makedirs(dst, exist_ok=True)
    for f in [os.path.basename(x) for x in glob.glob(os.path.join(os.path.dirname(mp), '*'))]:  # model.pt + all result json
        shutil.copy(os.path.join(os.path.dirname(mp), f), dst)
    n += 1
print(f'restored {n} trained models:', sorted(os.listdir('runs')))

## Config — which runs this session should execute

In [ ]:
EPOCHS = 100
WHERE  = 'block'             # 'head' = nn.Dropout before classifier (main study, done: seeds 0-2); 'block' = Dropout2d in residual blocks (E2)
SEEDS  = '0'                 # seeds to TRAIN this session for WHERE
GPU0_P = '0.0 0.1'           # dropout rates handled by GPU 0
GPU1_P = '0.3 0.5'           # dropout rates handled by GPU 1 (ignored if only one GPU)
RUN_FINETUNE = True          # E1 (one-shot vs iterative) on the runs matching FT_SEEDS / FT_WHERE at FT_SPARSITIES
FT_SEEDS = ['0']
FT_WHERE = 'head'            # which models to fine-tune ('head' or 'block')
FT_SPARSITIES = ['0.95', '0.98']   # 60/80/90% are done for head seed 0; push to the regime where schedules separate
os.makedirs('runs', exist_ok=True)

## Pilot (B3): smoke-test the pipeline for 1 epoch before committing hours

In [ ]:
if not glob.glob('runs/p*_s*/model.pt'):
    !python train.py --p 0.0 --seed 0 --epochs 1 --out runs_pilot && python prune_sweep.py --run runs_pilot/p0.0_s0 --skip_layerwise
else: print('models already present, skipping pilot')

## Dense training matrix + pruning sweeps (both GPUs in parallel)

In [ ]:
import torch, subprocess, time
ngpu = torch.cuda.device_count()
# 1) pruning sweeps for restored models that lack one (fast, ~5 min each)
restored = [d for d in sorted(glob.glob('runs/p*_s*')) if not os.path.exists(f'{d}/prune_sweep.json')]
for i in range(0, len(restored), max(ngpu,1)):
    ps = [subprocess.Popen(['python', 'prune_sweep.py', '--run', d], env=dict(os.environ, CUDA_VISIBLE_DEVICES=str(g)))
          for g, d in enumerate(restored[i:i+max(ngpu,1)])]
    for p in ps: p.wait()
# 2) dense training matrix for SEEDS (both GPUs in parallel)
procs = [subprocess.Popen(['bash', 'run_matrix.sh', GPU0_P, SEEDS, str(EPOCHS), '0'], env=dict(os.environ, WHERE=WHERE))]
if ngpu > 1:
    procs.append(subprocess.Popen(['bash', 'run_matrix.sh', GPU1_P, SEEDS, str(EPOCHS), '1'], env=dict(os.environ, WHERE=WHERE)))
else:
    procs[0].wait(); procs.append(subprocess.Popen(['bash', 'run_matrix.sh', GPU1_P, SEEDS, str(EPOCHS), '0'], env=dict(os.environ, WHERE=WHERE)))
for p in procs: p.wait()
!tail -n 1 runs/train_*.log

## E1: one-shot vs iterative at matched 20-epoch budget (FT_SPARSITIES)

In [ ]:
if RUN_FINETUNE:
    suf = 'blk' if FT_WHERE == 'block' else ''
    jobs = [(r, s, sch) for r in sorted(glob.glob(f'runs/p*{suf}_s*')) if r.split('_s')[-1] in FT_SEEDS and (('blk' in r) == (FT_WHERE == 'block'))
            for s in FT_SPARSITIES for sch in ['oneshot', 'iterative']
            if not os.path.exists(f'{r}/ft_{sch}_s{s}.json')]
    print(len(jobs), 'fine-tuning jobs')
    queues = [jobs[i::max(ngpu,1)] for i in range(max(ngpu,1))]
    def run_queue(q, gpu):
        env = dict(os.environ, CUDA_VISIBLE_DEVICES=str(gpu))
        cmd = ' ; '.join(f'python finetune.py --run {r} --sparsity {s} --schedule {sch}' for r, s, sch in q) or 'true'
        return subprocess.Popen(['bash', '-c', cmd], env=env)
    ps = [run_queue(q, g) for g, q in enumerate(queues)]
    for p in ps: p.wait()

## Figures + tables

In [ ]:
!pip -q install tabulate
!python plots.py --runs runs --out figures --where head
!python plots.py --runs runs --out figures_block --where block
!ls figures
!cd /kaggle/working && tar czf results_$(date +%Y%m%d_%H%M).tar.gz code/runs/*/*.json code/runs/*.log code/figures* && ls -la /kaggle/working/*.tar.gz
from IPython.display import Image, display
for f in ['acc_vs_sparsity.png', 'acc_vs_sparsity_bnrecal.png', 'finetune.png', 'weight_hist.png']:
    if os.path.exists('figures/' + f): display(Image('figures/' + f))